# Tehran Urban Heat Island (UHI) Analysis with Remote Sensing

**Goal:** Retrieve Land Surface Temperature (LST) from Landsat 8/9 Collection 2 Level-2 imagery, examine its relationship with vegetation (NDVI) and built-up density (NDBI), identify hot and cold thermal clusters using the Getis-Ord Gi* spatial statistic, and compare the urban heat island trend between summer 2015 and summer 2024.

**Data:**
- Landsat 8 & 9 Collection 2 Level-2 (surface reflectance + brightness temperature), via Google Earth Engine
- ESA WorldCover v200 (10 m global land cover map, 2021)

**Method:** Mono-window algorithm with NDVI-based emissivity correction for LST retrieval; 500 m grid sampling; Pearson correlation; Getis-Ord Gi* spatial autocorrelation analysis.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.gee_setup import init_earth_engine, load_config, get_aoi
from src.indices import preprocess_collection
from src.lst import build_lst_composite
from src.lulc import load_worldcover
from src.uhi_analysis import (
    sample_grid_to_dataframe, to_geodataframe, correlation_report,
    getis_ord_hotspots, uhi_intensity,
)
from src.visualization import make_interactive_map, plot_lst_histogram, plot_ndvi_lst_scatter, plot_hotspot_map, plot_correlation_heatmap

cfg = load_config()
init_earth_engine(cfg['project']['ee_project_id'])
aoi = get_aoi(cfg)

## 1. Preprocessing and LST extraction for two time periods

In [ ]:
baseline_period = cfg['time_periods']['baseline']
recent_period = cfg['time_periods']['recent']

coll_baseline = preprocess_collection(
    cfg['landsat']['baseline_collection'], aoi,
    baseline_period['start'], baseline_period['end'], cfg['landsat']['cloud_cover_max'])
coll_recent = preprocess_collection(
    cfg['landsat']['recent_collection'], aoi,
    recent_period['start'], recent_period['end'], cfg['landsat']['cloud_cover_max'])

print('Number of 2015 scenes:', coll_baseline.size().getInfo())
print('Number of 2024 scenes:', coll_recent.size().getInfo())

lst_baseline = build_lst_composite(coll_baseline)
lst_recent = build_lst_composite(coll_recent)

## 2. Land cover map (ESA WorldCover)

In [ ]:
worldcover = load_worldcover(aoi, cfg['lulc']['collection'], cfg['lulc']['year'])

## 3. Interactive map (Leaflet) — comparing LST across both periods and NDVI

In [ ]:
m = make_interactive_map(aoi, lst_baseline, lst_recent, worldcover)
m

## 4. Grid sampling and conversion to a GeoDataFrame

In [ ]:
cell_size = cfg['analysis']['grid_cell_size_m']

df_baseline = sample_grid_to_dataframe(lst_baseline, aoi, cell_size)
gdf_baseline = to_geodataframe(df_baseline)

df_recent = sample_grid_to_dataframe(lst_recent, aoi, cell_size)
gdf_recent = to_geodataframe(df_recent)

gdf_recent.head()

## 5. Temperature distribution and warming trend between the two periods

In [ ]:
plot_lst_histogram(gdf_baseline, gdf_recent, '../outputs/figures/lst_histogram.png')
warming = gdf_recent['LST_C'].mean() - gdf_baseline['LST_C'].mean()
print(f"Mean LST 2015: {gdf_baseline['LST_C'].mean():.2f} °C")
print(f"Mean LST 2024: {gdf_recent['LST_C'].mean():.2f} °C")
print(f"Warming over 9 years: {warming:.2f} °C")

## 6. Relationship between temperature, vegetation, and built-up density

In [ ]:
corr = correlation_report(gdf_recent)
plot_correlation_heatmap(corr, '../outputs/figures/correlation_heatmap.png')
plot_ndvi_lst_scatter(gdf_recent, '../outputs/figures/ndvi_lst_scatter.png')
corr

## 7. Identifying hot and cold spots (Getis-Ord Gi*)

In [ ]:
gdf_hotspots = getis_ord_hotspots(gdf_recent, value_col='LST_C')
plot_hotspot_map(gdf_hotspots, '../outputs/figures/hotspot_map.png')
gdf_hotspots['hotspot_class'].value_counts()

## 8. Conclusion

This notebook's outputs (maps, charts, and CSV tables) are saved under `outputs/` and form the basis of the project's final report in `docs/methodology.md` and `README.md`.

For a fully reproducible run against real, pre-selected Landsat scenes that requires no Google Earth Engine account at all, see `scripts/fetch_real_scenes.py` and `scripts/analyze_real_scenes.py`, which apply the same algorithms to imagery downloaded directly from the public Microsoft Planetary Computer catalog.